In [13]:
# Significant similarity to audio descriptions. 

import numpy as np
import pandas as pd

scores = np.load('../scores/S1/imagined_speech/gpt_layer_10/alpha_repeat-1.npz', allow_pickle=True)
print(scores.files)
score_names = ['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
score_files = {}
for name in score_names:
    score_files[name] = scores[name].item()

print(score_files['story_zscores'])

['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
{('alpha_repeat-1', 'WER'): np.float64(0.6803876542976238), ('alpha_repeat-1', 'BLEU'): np.float64(1.1852455224633727), ('alpha_repeat-1', 'METEOR'): np.float64(1.8014219498921065), ('alpha_repeat-1', 'BERT'): np.float32(1.7755879)}


In [14]:
window_zscores = {'subject': [], 'gpt_layer': [], 'WER':[],'BLEU':[], 'METEOR':[], 'BERT':[]}
for subject in ['S1','S2','S3']:
    for gpt_layer in [6,7,8,9,10]:
        for task in ['alpha_repeat-1']:
            scores = np.load(f'../scores/{subject}/imagined_speech/gpt_layer_{gpt_layer}/{task}.npz', allow_pickle=True)['window_zscores'].item()
            # print(scores['window_zscores'].item())
            window_zscores['subject'].append(subject)
            window_zscores['gpt_layer'].append(gpt_layer)
            window_zscores['WER'].append(scores[(task, 'WER')])
            window_zscores['BLEU'].append(scores[(task, 'BLEU')])
            window_zscores['METEOR'].append(scores[(task, 'METEOR')])
            window_zscores['BERT'].append(scores[(task, 'BERT')])

window_zscores

{'subject': ['S1',
  'S1',
  'S1',
  'S1',
  'S1',
  'S2',
  'S2',
  'S2',
  'S2',
  'S2',
  'S3',
  'S3',
  'S3',
  'S3',
  'S3'],
 'gpt_layer': [6, 7, 8, 9, 10, 6, 7, 8, 9, 10, 6, 7, 8, 9, 10],
 'WER': [array([-0.04627448, -0.41003426, -0.03793216, -0.08561877,  0.42254826,
          0.46447889, -0.30012559,  0.46271912,  0.19150803,  0.58511751,
          1.94550548,  1.85176193,  1.54195294,  1.93717165,  1.54088314,
          1.99043633,  1.84925535,  2.0331765 ,  1.86815358,  1.88082761,
          1.94602231,  1.95065206,  1.54499124,  2.43862888,  1.85616901,
          1.90264675,  1.1247263 ,  0.77132494,  0.98285415,  0.99669791,
          1.00169525,  1.07539739,  0.9486833 ,  1.2829225 ,  0.83317896,
          0.33159634, -0.30467346, -0.27147318,  0.53207781,  0.66750044,
          0.5535254 ]),
  array([-0.93286234, -0.37969035,  0.01561928, -0.040996  , -0.34291383,
          0.51438453,  0.54922502, -0.19602825,  0.27216553, -0.02322095,
          0.55735497,  0.4493758 

In [15]:
results_df = pd.DataFrame(window_zscores)

print(len(results_df.loc[0, 'WER']))
print(len(results_df.loc[1, 'WER']))
print(len(results_df.loc[2, 'WER']))
print('=')
print(563*3)
m=1689

from scipy.stats import norm
#convert to p val
results_df['WER'] = results_df['WER'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1-norm.cdf(z) for z in l])
#sort
for i in range(3):
    for met in ['WER','BLEU','METEOR','BERT']:
        results_df.loc[i,met].sort()
# to q and check threshold
def p_to_q(p, i):
    return p*m/i
results_df['WER'] = results_df['WER'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])

# S1=np.array(results_df.loc[0, 'BERT'] + results_df.loc[1, 'BERT'] + results_df.loc[2, 'BERT']).mean()
# S2=np.array(results_df.loc[3, 'BERT'] + results_df.loc[4, 'BERT'] + results_df.loc[5, 'BERT']).mean()
# S3=np.array(results_df.loc[6, 'BERT'] + results_df.loc[7, 'BERT'] + results_df.loc[8, 'BERT']).mean()
# print(S1,S2,S3,'BERT')

results_df

41
41
41
=
1689


,subject,gpt_layer,WER,BLEU,METEOR,BERT
0,S1,6,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,S1,7,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,S1,8,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, ..."
3,S1,9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,S1,10,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,S2,6,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
6,S2,7,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7,S2,8,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
8,S2,9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
9,S2,10,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [16]:
results_df['BERT'] = results_df['BERT'].apply(np.mean)
to_file = results_df.drop(columns=['WER','BLEU','METEOR']).rename(columns={'BERT':'significantly_decoded'})
to_file

,subject,gpt_layer,significantly_decoded
0,S1,6,0.000000
1,S1,7,0.414634
2,S1,8,0.341463
3,S1,9,0.341463
4,S1,10,0.024390
5,S2,6,0.000000
6,S2,7,0.024390
7,S2,8,0.219512
8,S2,9,0.000000
9,S2,10,0.024390


In [17]:
# gpt_layer_6=np.array(results_df.loc[0, 'BERT']).mean()
# gpt_layer_7=np.array(results_df.loc[1, 'BERT']).mean()
# gpt_layer_8=np.array(results_df.loc[2, 'BERT']).mean()
# gpt_layer_9=np.array(results_df.loc[3, 'BERT']).mean()
# gpt_layer_10=np.array(results_df.loc[4, 'BERT']).mean()
# to_file = pd.DataFrame({'gpt_layer':[6,7,8,9,10], 'significantly_decoded': [gpt_layer_6,gpt_layer_7,gpt_layer_8,gpt_layer_9,gpt_layer_10]})
to_file.to_csv('imagined_speech_gpt_layer.csv', index=False)

to_file

,subject,gpt_layer,significantly_decoded
0,S1,6,0.000000
1,S1,7,0.414634
2,S1,8,0.341463
3,S1,9,0.341463
4,S1,10,0.024390
5,S2,6,0.000000
6,S2,7,0.024390
7,S2,8,0.219512
8,S2,9,0.000000
9,S2,10,0.024390
